# Gold volatility baseline forecasts

This notebook loads the prepared gold-futures dataset, computes baseline variance forecasts at each forecast origin, and compares each model with the actual realized-variance target. Each model gets its own chart panel, followed by MAE/RMSE summary metrics.

In [ ]:
from pathlib import Path

from gold_vol_pipeline import prediction_history

repo_root = Path.cwd()
if not (repo_root / "data" / "processed" / "gold_futures.csv").exists():
    repo_root = repo_root.parent
data_path = repo_root / "data" / "processed" / "gold_futures.csv"
plot_dir = repo_root / "plots" / "model_comparisons"
print(f"Using data file: {data_path}")


In [ ]:
import pandas as pd

df = pd.read_csv(data_path)
df.head()


In [ ]:
comparison = prediction_history(
    str(data_path),
    horizon=5,
    sample_size=500,
)
comparison.head()


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display

plot_dir.mkdir(parents=True, exist_ok=True)
model_columns = [
    'historical_variance',
    'rolling_variance',
    'ewma_variance',
    'ar1_squared_return',
    'ma1_squared_return',
    'arch1',
    'garch11',
]
metrics = pd.DataFrame([
    {
        'model': model,
        'MAE': (comparison[model] - comparison['actual']).abs().mean(),
        'RMSE': ((comparison[model] - comparison['actual']) ** 2).mean() ** 0.5,
    }
    for model in model_columns
]).sort_values('RMSE')
display(metrics)

plot_paths = []
for model in model_columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(comparison['date'], comparison['actual'], label='actual', color='black', linewidth=1.5)
    ax.plot(comparison['date'], comparison[model], label=f'{model} forecast', alpha=0.85)
    ax.set_title(f'Actual vs {model} forecast (5-session realized variance)')
    ax.set_xlabel('Forecast origin date')
    ax.set_ylabel('Realized variance')
    ax.legend(loc='upper right')
    plt.xticks(rotation=45)
    fig.tight_layout()
    output_path = plot_dir / f'{model}_vs_actual.png'
    fig.savefig(output_path, dpi=200)
    plt.close(fig)
    plot_paths.append(output_path)
    display(Image(filename=str(output_path)))

print('Saved comparison charts:')
for output_path in plot_paths:
    print(output_path)
